# 🤖 Pota Bot: costruiamo un agente AI, dal concetto al codice

**GDG Brescia · Hands-on · giovedì 1 ottobre 2026 · CSMT Innovation Hub**

> *Un modello risponde. Un agente risolve.*

Stasera costruiamo **Pota Bot**, il bot della community GDG Brescia, con **Google ADK** (Agent Development Kit) e **Gemini**.
Il codice è open source: [github.com/GDGBrescia/pota-bot](https://github.com/GDGBrescia/pota-bot). Chi c'era stasera può dire *"l'ho costruito io"*.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GDGBrescia/pota-bot/blob/main/workshop.ipynb)

## Cosa costruirai

Un agente che:
- risponde sugli **eventi** e sulle **domande frequenti** della community;
- porta le **notizie su AI e Google** nelle categorie che avete scelto voi;
- **raccoglie proposte** di temi per i prossimi eventi;
- lavora **dentro dei guardrail**: regole scritte nel codice che non può aggirare.

```
Utente ──► Pota Bot (Gemini + ADK)
              │  instruction: chi è, che tono ha, cosa NON fa
              ├─ get_upcoming_events()      → eventi GDG Brescia
              ├─ get_faq(topic)             → domande frequenti
              ├─ get_news(category)         → notizie per categoria
              └─ propose_topic(title, why)  → salva una proposta (AZIONE)
                     ▲
                     └─ guardrail: blocca dati personali, max 3 proposte
```

## Perché proprio questo bot? L'avete chiesto voi

Ad agosto abbiamo chiesto nel gruppo Telegram *"Cosa può fare per te la community?"*. **27 voti**:

| Categoria | Voti |
|---|---|
| 🎟️ Eventi, hackathon e opportunità | 66% |
| 🧠 Nuovi modelli AI | 59% |
| 🛠️ Strumenti per sviluppatori | 59% |
| 💼 AI per il business e casi d'uso in azienda | 59% |
| 🔵 Novità dell'ecosistema Google | 37% |
| 📄 Ricerche e paper spiegati semplici | 33% |
| 🙅 Non fa per me | 0% |

Stasera il bot impara prima di tutto la cosa più votata: **gli eventi**. Le 6 categorie diventano le categorie delle notizie.

## Cosa imparerai: i 4 ingredienti di un agente

1. **Istruzioni**: chi è e come si comporta.
2. **Strumenti**: cosa sa fare oltre a "parlare".
3. **Stato**: cosa si ricorda mentre lavora.
4. **Guardrail**: cosa non deve fare, anche se glielo chiedi.

## Cosa ti serve

- Un browser e un **account Google personale** (gli account aziendali o scolastici a volte hanno AI Studio disattivato).
- Una **chiave API Gemini** gratuita da [AI Studio](https://aistudio.google.com/apikey).
- Niente da installare.

## Come funziona questo notebook

In ogni step trovi questi simboli:

| Simbolo | Cosa significa |
|---|---|
| 🎯 **Obiettivo** | Cosa ottieni alla fine dello step |
| 📖 **Capire** | Il concetto, in parole semplici. **Leggilo prima del codice**: è la parte che conta |
| ✍️ **Tocca a te** | La parte che scrivi tu: istruzioni, descrizioni, regole. È quella che *guida* l'agente |
| 🔍 **Osserva** | Cosa guardare nel risultato |
| 💼 **In azienda** | Lo stesso concetto applicato al lavoro vero |
| 🚀 **Sfida** | Per chi va veloce, facoltativa |
| 🛟 **Salta qui** | Se sei rimasto indietro, una cella che ti rimette in pari |

**Tre livelli:**
- 🟢 **Base** (step 1–6): lo facciamo tutti insieme stasera.
- 🟡 **Extra**: per chi finisce prima. Il **digest settimanale** di notizie.
- 🔵 **Avanzato**: da fare a casa. Telegram, multi-agente, contribuire al repo.

> 💡 **Usi l'AI per programmare? Benissimo.** Colab ha Gemini integrato (l'icona ✨), e il codice di questo notebook si scrive in pochi secondi.
> Proprio per questo la parte importante di stasera non è battere codice ma **capire** cosa succede e **decidere** come guidare l'agente.
> Le parti ✍️ puoi scriverle anche con l'AI, ma **decidi tu** cosa devono dire.

---
# ⚙️ Setup (5 minuti, meglio se a casa)

1. **Salva una copia:** menu *File → Salva una copia su Drive*. Lavorerai sulla tua copia.
2. **Crea la chiave API:** vai su [aistudio.google.com/apikey](https://aistudio.google.com/apikey) → *Create API key*. Copiala.
3. **Mettila nei Secrets di Colab:** clicca sull'icona 🔑 nella barra a sinistra → *Aggiungi nuovo secret*.
   - Nome: `GOOGLE_API_KEY`
   - Valore: la tua chiave
   - Attiva **Accesso notebook**.
4. Esegui le celle qui sotto (▶️ oppure `Shift+Invio`). La prima **scarica il repo di Pota Bot** (il codice completo e i dati) e installa ADK. Se alla fine vedi ✅, sei pronto.

> 🔒 **La chiave è come una password.** Non incollarla mai nel codice, in una chat o in uno screenshot. I Secrets servono proprio a questo. Se per errore la mostri, revocala in AI Studio a fine serata.

In [ ]:
#@title 📦 Scarica il repo di Pota Bot e installa Google ADK
import os
if not os.path.exists("pota_bot/data"):          # su Colab il repo non c'è ancora: lo scarichiamo
    if not os.path.exists("pota-bot"):
        !git clone -q https://github.com/GDGBrescia/pota-bot.git
    %cd pota-bot
%pip install -q -r requirements.txt
print("✅ Repo scaricato e ADK installato")

In [ ]:
#@title 🔑 Configurazione: chiave, modello e prova (esegui e aspetta il ✅)
import os, warnings
warnings.filterwarnings("ignore")

from google.colab import userdata
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    raise RuntimeError(
        "❌ Non trovo la chiave. Controlla nei Secrets (🔑 a sinistra) che si chiami esattamente "
        "GOOGLE_API_KEY e che 'Accesso notebook' sia attivo."
    ) from None
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

from google import genai
from google.genai import errors as genai_errors

# ⏳ Matteo: fissare qui l'ID del modello dopo aver letto le quote in AI Studio.
# Se resta vuoto, il notebook sceglie da solo un modello Flash-Lite disponibile con la tua chiave.
MODEL_SCELTO = ""

client = genai.Client()
try:
    disponibili = [m.name.removeprefix("models/") for m in client.models.list()
                   if "generateContent" in (m.supported_actions or [])]
    flash_lite = sorted(n for n in disponibili if "flash-lite" in n and "preview" not in n)
    MODEL = MODEL_SCELTO or (flash_lite[-1] if flash_lite else None)
    if MODEL is None:
        raise RuntimeError("❌ Nessun modello Flash-Lite disponibile con questa chiave. Chiedi aiuto ai tavoli.")
    risposta = client.models.generate_content(model=MODEL, contents="Rispondi solo con: pronto")
    print(f"✅ Tutto pronto! Modello: {MODEL}. Il modello dice: {risposta.text.strip()}")
except genai_errors.ClientError as e:
    if e.code == 429:
        print("⏳ Troppe richieste (429). Aspetta un minuto e riesegui la cella.")
    elif e.code in (400, 401, 403):
        print("❌ La chiave non funziona. Ricopiala da AI Studio nei Secrets e riprova.")
    else:
        print(f"❌ Errore {e.code}: {e.message}")
    raise

In [ ]:
#@title 🧰 Funzione di aiuto: chiedi()
async def chiedi(runner, messaggio, sessione="workshop"):
    """Manda un messaggio all'agente e mostra cosa succede, comprese le chiamate agli strumenti."""
    try:
        return await runner.run_debug(messaggio, session_id=sessione, verbose=True)
    except genai_errors.ClientError as e:
        if e.code == 429:
            print("⏳ Troppe richieste (429). Aspetta un minuto e riprova.\n"
                  "Se succede ancora, la quota di oggi potrebbe essere finita: "
                  "cambia MODEL_SCELTO nella cella di configurazione, oppure lavora in coppia con il vicino.")
        else:
            print(f"❌ Errore dal modello ({e.code}): {e.message}")

print("✅ chiedi() pronta")

📖 **Cos'è `chiedi()`?** Una scorciatoia per non riscrivere ogni volta le stesse righe. Dentro chiama `runner.run_debug(...)` di ADK con `verbose=True`, cioè: *"mostrami anche quali strumenti usa l'agente"*. Se qualcosa va storto, ti spiega cosa fare in italiano.

---
# 🟢 Step 1: Un modello risponde

🎯 **Obiettivo:** vedere cosa succede quando un modello non ha strumenti.

📖 **Capire.** Un **LLM** (*Large Language Model*, come Gemini) è bravissimo a produrre testo plausibile, ma sa solo quello che ha visto durante l'addestramento. Non conosce i fatti di oggi e non può "andare a vedere". Quando non sa, spesso **inventa con sicurezza**: si chiama *allucinazione*.

In **ADK** i pezzi base sono due:
- `Agent`: l'agente. Ha un nome, un modello e delle istruzioni (per ora niente strumenti).
- `Runner`: il "motore" che fa girare l'agente e tiene traccia della conversazione.

In [ ]:
from typing import Literal
import re
from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import ToolContext

modello_nudo = Agent(
    name="modello_nudo",
    model=MODEL,
    instruction="Sei un assistente utile.",
)
runner = InMemoryRunner(agent=modello_nudo)

# ✍️ TOCCA A TE: cambia la domanda, se vuoi.
await runner.run_debug("Quando è il prossimo evento di GDG Brescia?")

🔍 **Osserva:** risponde in modo vago, o si inventa una data? In ogni caso **non lo sa**, e spesso non lo dice.

💼 **In azienda:** è il motivo n.1 per cui un chatbot "generico" non basta. Non conosce i tuoi clienti, i tuoi prodotti, i tuoi processi.

🚀 **Sfida:** chiedigli le notizie AI di oggi.

---
# 🟢 Step 2: Il primo agente

🎯 **Obiettivo:** dare all'agente un'identità e il primo strumento.

📖 **Capire.** Uno **strumento** (*tool*) è una normale funzione Python che l'agente può decidere di chiamare. Come fa il modello a sapere che esiste e a cosa serve? ADK gli mostra tre cose:
- il **nome** della funzione;
- la **docstring**, cioè il testo tra `"""` sotto la definizione: è la descrizione che il modello legge;
- i **tipi** degli argomenti.

Poi funziona così: *domanda → il modello decide "mi serve lo strumento X" → ADK esegue la funzione → il risultato torna al modello → il modello risponde*. È il **ciclo dell'agente**, e **è il modello a decidere** quando usare uno strumento.

Le **istruzioni** (`instruction`) sono il "carattere" dell'agente: chi è, che tono usa, cosa deve e non deve fare.

📦 Prima i dati della community: eventi, FAQ e notizie. Stanno nei file JSON del repo che hai appena scaricato (cartella `pota_bot/data/`, dalla barra 📁 a sinistra).

In [ ]:
# 📦 I dati della community: stanno nei file JSON del repo, in pota_bot/data/.
# Aprili dalla barra a sinistra (📁) e modificali come vuoi, poi riesegui questa cella.
import json
from pathlib import Path

DATA_DIR = Path("pota_bot/data")
EVENTS = json.loads((DATA_DIR / "events.json").read_text(encoding="utf-8"))
FAQ = json.loads((DATA_DIR / "faq.json").read_text(encoding="utf-8"))
NEWS = json.loads((DATA_DIR / "news.json").read_text(encoding="utf-8"))  # le 6 categorie del sondaggio
print(f"📦 {len(EVENTS)} eventi, {len(FAQ)} FAQ, notizie in {len(NEWS)} categorie")

✍️ **Tocca a te:** le istruzioni del bot. Parti da queste e cambiale come vuoi: tono, regole, cosa deve rifiutare.

In [ ]:
# ✍️ TOCCA A TE: questa è la "personalità" del bot. Riscrivila a modo tuo!
INSTRUCTION = """
Sei Pota Bot, l'assistente della community GDG Brescia (Google Developer Group).

Tono: diretto, pratico e amichevole, con un pizzico di bresciano ("Pota!"). Risposte brevi.

Regole:
- Per i fatti (date, luoghi, notizie) usa SEMPRE gli strumenti che hai. Non inventare.
- Se non hai l'informazione, dillo chiaramente.
- Rispondi in italiano.
"""

In [ ]:
def get_upcoming_events() -> dict:
    """Restituisce gli eventi di GDG Brescia (passati e futuri) con data, titolo e luogo."""
    return {"events": EVENTS}

pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events],
)
runner = InMemoryRunner(agent=pota_bot)

await chiedi(runner, "Quando è il prossimo evento?")

🔍 **Osserva:** nell'output compare `[Calling tool: get_upcoming_events(...)]`. È il modello che ha **deciso** di usare lo strumento: nessuno gliel'ha ordinato.

💼 **In azienda:** l'agente legge i **tuoi** dati (gestionale, CRM, database), non quelli di internet. È qui che un chatbot diventa utile.

🚀 **Sfida:** cambia la docstring di `get_upcoming_events` in qualcosa di vago (es. `"""Fa cose."""`), riesegui la cella: lo usa ancora?

---
# 🟢 Step 3: L'agente sceglie

🎯 **Obiettivo:** dare più strumenti all'agente e vedere come sceglie quello giusto.

🛟 **Salta qui**: sei rimasto indietro o sei appena arrivato? Esegui la cella qui sotto: ricostruisce tutto fino all'inizio dello step 3.
*(Serve comunque aver eseguito il Setup. Se hai già fatto gli step precedenti, puoi saltarla.)*

In [ ]:
#@title 🛟 Salta qui: tutto pronto per lo step 3
from typing import Literal
import re
from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import ToolContext
# 📦 I dati della community: stanno nei file JSON del repo, in pota_bot/data/.
# Aprili dalla barra a sinistra (📁) e modificali come vuoi, poi riesegui questa cella.
import json
from pathlib import Path

DATA_DIR = Path("pota_bot/data")
EVENTS = json.loads((DATA_DIR / "events.json").read_text(encoding="utf-8"))
FAQ = json.loads((DATA_DIR / "faq.json").read_text(encoding="utf-8"))
NEWS = json.loads((DATA_DIR / "news.json").read_text(encoding="utf-8"))  # le 6 categorie del sondaggio
print(f"📦 {len(EVENTS)} eventi, {len(FAQ)} FAQ, notizie in {len(NEWS)} categorie")
# ✍️ TOCCA A TE: questa è la "personalità" del bot. Riscrivila a modo tuo!
INSTRUCTION = """
Sei Pota Bot, l'assistente della community GDG Brescia (Google Developer Group).

Tono: diretto, pratico e amichevole, con un pizzico di bresciano ("Pota!"). Risposte brevi.

Regole:
- Per i fatti (date, luoghi, notizie) usa SEMPRE gli strumenti che hai. Non inventare.
- Se non hai l'informazione, dillo chiaramente.
- Rispondi in italiano.
"""
def get_upcoming_events() -> dict:
    """Restituisce gli eventi di GDG Brescia (passati e futuri) con data, titolo e luogo."""
    return {"events": EVENTS}

pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events],
)
runner = InMemoryRunner(agent=pota_bot)
print("🛟 Sei allineato all'inizio dello step 3. Puoi continuare da qui.")

📖 **Capire.** Con più strumenti, l'agente deve **scegliere**. Lo fa leggendo nomi e docstring: la docstring è un **contratto** tra te e il modello. Se è chiara, sceglie bene; se è vaga, sbaglia.

Guarda i due nuovi strumenti:
- `get_faq(topic: str)`: l'argomento è **libero**, il modello scrive quello che vuole.
- `get_news(category: Category)`: l'argomento è **a valori chiusi**. `Literal[...]` dice: *"può essere solo una di queste 6 categorie"*, cioè le categorie del sondaggio. Il modello **non può inventarsene una**.

✍️ **Tocca a te:** riscrivi la docstring di `get_faq`. Deve far capire al modello *quando* usarla.

In [ ]:
def get_faq(topic: str) -> dict:
    """✍️ TOCCA A TE: riscrivi questa docstring. È quello che il modello legge per decidere se usare lo strumento.

    Cerca nelle domande frequenti della community GDG Brescia.

    Args:
        topic: l'argomento della domanda in poche parole, ad es. "costi" o "proporre un talk".
    """
    topic = topic.lower()  # confronto per "radici" di parola: "cost" trova costo, costi, costa
    trovate = [f for f in FAQ if any(p in topic for p in f["parole_chiave"])]
    if not trovate:
        return {"trovate": [], "argomenti_disponibili": [f["domanda"] for f in FAQ]}
    return {"trovate": [{"domanda": f["domanda"], "risposta": f["risposta"]} for f in trovate]}

In [ ]:
Category = Literal["eventi_e_opportunita", "modelli_ai", "strumenti_dev",
                   "ai_business", "ecosistema_google", "paper"]

def get_news(category: Category) -> dict:
    """Restituisce le notizie della settimana su AI e Google per una categoria.

    Args:
        category: la categoria più adatta alla domanda. Le categorie sono quelle
            scelte dalla community nel sondaggio Telegram.
    """
    return {"category": category, "news": NEWS[category]}

In [ ]:
pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events, get_faq, get_news],
)
runner = InMemoryRunner(agent=pota_bot)

await chiedi(runner, "Quanto costa partecipare agli eventi?")
await chiedi(runner, "Novità sui nuovi modelli AI questa settimana?")
await chiedi(runner, "Quando ci vediamo la prossima volta?")

🔍 **Osserva:** tre domande, tre strumenti diversi. Nella chiamata a `get_news`, che categoria ha scelto il modello? Da cosa l'ha capito?

💼 **In azienda:** un agente vale quanto gli strumenti che gli dai. E scegliere la categoria giusta è esattamente **smistare una richiesta al reparto giusto**.

🚀 **Sfida:** fai una domanda a cavallo di due categorie (es. *"novità Google per chi programma"*). Poi aggiungi una FAQ nuova in `pota_bot/data/faq.json`, riesegui la cella dei dati e prova a fargliela trovare.

---
# 🟢 Step 4: Azione + stato

🎯 **Obiettivo:** far **fare** qualcosa all'agente, non solo rispondere.

🛟 **Salta qui**: sei rimasto indietro o sei appena arrivato? Esegui la cella qui sotto: ricostruisce tutto fino all'inizio dello step 4.
*(Serve comunque aver eseguito il Setup. Se hai già fatto gli step precedenti, puoi saltarla.)*

In [ ]:
#@title 🛟 Salta qui: tutto pronto per lo step 4
from typing import Literal
import re
from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import ToolContext
# 📦 I dati della community: stanno nei file JSON del repo, in pota_bot/data/.
# Aprili dalla barra a sinistra (📁) e modificali come vuoi, poi riesegui questa cella.
import json
from pathlib import Path

DATA_DIR = Path("pota_bot/data")
EVENTS = json.loads((DATA_DIR / "events.json").read_text(encoding="utf-8"))
FAQ = json.loads((DATA_DIR / "faq.json").read_text(encoding="utf-8"))
NEWS = json.loads((DATA_DIR / "news.json").read_text(encoding="utf-8"))  # le 6 categorie del sondaggio
print(f"📦 {len(EVENTS)} eventi, {len(FAQ)} FAQ, notizie in {len(NEWS)} categorie")
# ✍️ TOCCA A TE: questa è la "personalità" del bot. Riscrivila a modo tuo!
INSTRUCTION = """
Sei Pota Bot, l'assistente della community GDG Brescia (Google Developer Group).

Tono: diretto, pratico e amichevole, con un pizzico di bresciano ("Pota!"). Risposte brevi.

Regole:
- Per i fatti (date, luoghi, notizie) usa SEMPRE gli strumenti che hai. Non inventare.
- Se non hai l'informazione, dillo chiaramente.
- Rispondi in italiano.
"""
def get_upcoming_events() -> dict:
    """Restituisce gli eventi di GDG Brescia (passati e futuri) con data, titolo e luogo."""
    return {"events": EVENTS}
def get_faq(topic: str) -> dict:
    """✍️ TOCCA A TE: riscrivi questa docstring. È quello che il modello legge per decidere se usare lo strumento.

    Cerca nelle domande frequenti della community GDG Brescia.

    Args:
        topic: l'argomento della domanda in poche parole, ad es. "costi" o "proporre un talk".
    """
    topic = topic.lower()  # confronto per "radici" di parola: "cost" trova costo, costi, costa
    trovate = [f for f in FAQ if any(p in topic for p in f["parole_chiave"])]
    if not trovate:
        return {"trovate": [], "argomenti_disponibili": [f["domanda"] for f in FAQ]}
    return {"trovate": [{"domanda": f["domanda"], "risposta": f["risposta"]} for f in trovate]}
Category = Literal["eventi_e_opportunita", "modelli_ai", "strumenti_dev",
                   "ai_business", "ecosistema_google", "paper"]

def get_news(category: Category) -> dict:
    """Restituisce le notizie della settimana su AI e Google per una categoria.

    Args:
        category: la categoria più adatta alla domanda. Le categorie sono quelle
            scelte dalla community nel sondaggio Telegram.
    """
    return {"category": category, "news": NEWS[category]}

pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events, get_faq, get_news],
)
runner = InMemoryRunner(agent=pota_bot)
print("🛟 Sei allineato all'inizio dello step 4. Puoi continuare da qui.")

📖 **Capire.** Finora gli strumenti *leggevano*. Ora uno *agisce*: `propose_topic` **salva** una proposta. Le azioni sono più delicate delle letture, perché cambiano qualcosa nel mondo.

Dove salva? Nello **stato della sessione** (`tool_context.state`): un piccolo "blocco note" che resta per tutta la conversazione. Nota che `tool_context` il modello **non lo vede**: ADK lo passa da solo alla funzione.

In [ ]:
def propose_topic(title: str, reason: str, tool_context: ToolContext) -> dict:
    """Registra una proposta di tema per un prossimo evento di GDG Brescia.

    Args:
        title: il tema proposto, in poche parole.
        reason: perché interessa alla community.
    """
    # ✍️ TOCCA A TE: cosa vuoi salvare di ogni proposta? (es. anche la data, o una categoria)
    proposals = tool_context.state.get("proposals", [])
    proposals.append({"title": title, "reason": reason})
    tool_context.state["proposals"] = proposals
    tool_context.state["proposals_count"] = len(proposals)
    return {"status": "salvata", "totale_proposte": len(proposals)}

pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events, get_faq, get_news, propose_topic],
)
runner = InMemoryRunner(agent=pota_bot)

# ✍️ TOCCA A TE: proponi un tema vero per il prossimo evento!
await chiedi(runner, "Vorrei proporre un tema: agenti AI per il customer care. Interessa a molte aziende qui.")

Ora guardiamo lo **stato** della sessione: la proposta è rimasta salvata.

In [ ]:
sessione = await runner.session_service.get_session(
    app_name=runner.app_name, user_id="debug_user_id", session_id="workshop")
sessione.state

🔍 **Osserva:** l'agente ha **fatto** qualcosa, e il risultato è rimasto nello stato.

💼 **In azienda:** è il passaggio da "risponde" a "apre un ticket", "registra una richiesta", "prenota".

🌱 **La tua proposta conta:** se hai proposto un tema vero, scrivilo anche nel gruppo Telegram. Li leggiamo tutti.

🚀 **Sfida:** proponi due temi in una sola frase. Li salva entrambi?

---
# 🟢 Step 5: Guardrail

🎯 **Obiettivo:** mettere dei limiti che l'agente **non può aggirare**.

🛟 **Salta qui**: sei rimasto indietro o sei appena arrivato? Esegui la cella qui sotto: ricostruisce tutto fino all'inizio dello step 5.
*(Serve comunque aver eseguito il Setup. Se hai già fatto gli step precedenti, puoi saltarla.)*

In [ ]:
#@title 🛟 Salta qui: tutto pronto per lo step 5
from typing import Literal
import re
from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import ToolContext
# 📦 I dati della community: stanno nei file JSON del repo, in pota_bot/data/.
# Aprili dalla barra a sinistra (📁) e modificali come vuoi, poi riesegui questa cella.
import json
from pathlib import Path

DATA_DIR = Path("pota_bot/data")
EVENTS = json.loads((DATA_DIR / "events.json").read_text(encoding="utf-8"))
FAQ = json.loads((DATA_DIR / "faq.json").read_text(encoding="utf-8"))
NEWS = json.loads((DATA_DIR / "news.json").read_text(encoding="utf-8"))  # le 6 categorie del sondaggio
print(f"📦 {len(EVENTS)} eventi, {len(FAQ)} FAQ, notizie in {len(NEWS)} categorie")
# ✍️ TOCCA A TE: questa è la "personalità" del bot. Riscrivila a modo tuo!
INSTRUCTION = """
Sei Pota Bot, l'assistente della community GDG Brescia (Google Developer Group).

Tono: diretto, pratico e amichevole, con un pizzico di bresciano ("Pota!"). Risposte brevi.

Regole:
- Per i fatti (date, luoghi, notizie) usa SEMPRE gli strumenti che hai. Non inventare.
- Se non hai l'informazione, dillo chiaramente.
- Rispondi in italiano.
"""
def get_upcoming_events() -> dict:
    """Restituisce gli eventi di GDG Brescia (passati e futuri) con data, titolo e luogo."""
    return {"events": EVENTS}
def get_faq(topic: str) -> dict:
    """✍️ TOCCA A TE: riscrivi questa docstring. È quello che il modello legge per decidere se usare lo strumento.

    Cerca nelle domande frequenti della community GDG Brescia.

    Args:
        topic: l'argomento della domanda in poche parole, ad es. "costi" o "proporre un talk".
    """
    topic = topic.lower()  # confronto per "radici" di parola: "cost" trova costo, costi, costa
    trovate = [f for f in FAQ if any(p in topic for p in f["parole_chiave"])]
    if not trovate:
        return {"trovate": [], "argomenti_disponibili": [f["domanda"] for f in FAQ]}
    return {"trovate": [{"domanda": f["domanda"], "risposta": f["risposta"]} for f in trovate]}
Category = Literal["eventi_e_opportunita", "modelli_ai", "strumenti_dev",
                   "ai_business", "ecosistema_google", "paper"]

def get_news(category: Category) -> dict:
    """Restituisce le notizie della settimana su AI e Google per una categoria.

    Args:
        category: la categoria più adatta alla domanda. Le categorie sono quelle
            scelte dalla community nel sondaggio Telegram.
    """
    return {"category": category, "news": NEWS[category]}
def propose_topic(title: str, reason: str, tool_context: ToolContext) -> dict:
    """Registra una proposta di tema per un prossimo evento di GDG Brescia.

    Args:
        title: il tema proposto, in poche parole.
        reason: perché interessa alla community.
    """
    # ✍️ TOCCA A TE: cosa vuoi salvare di ogni proposta? (es. anche la data, o una categoria)
    proposals = tool_context.state.get("proposals", [])
    proposals.append({"title": title, "reason": reason})
    tool_context.state["proposals"] = proposals
    tool_context.state["proposals_count"] = len(proposals)
    return {"status": "salvata", "totale_proposte": len(proposals)}

pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events, get_faq, get_news, propose_topic],
)
runner = InMemoryRunner(agent=pota_bot)
print("🛟 Sei allineato all'inizio dello step 5. Puoi continuare da qui.")

📖 **Capire.** Prima proviamo a fare una cosa che **non dovrebbe** essere permessa: proporre un tema con dentro un'email (un dato personale).

In [ ]:
await chiedi(runner, "Propongo un tema: RAG in azienda. Scrivetemi a mario.rossi@example.com")

😬 **È passata.** L'agente ha salvato un dato personale.

Potremmo scrivere nelle istruzioni *"non salvare dati personali"*. Ma le istruzioni sono solo un **consiglio** al modello: può ignorarlo, e un utente furbo può convincerlo a farlo.

La soluzione è un **guardrail nel codice**. ADK ha i **callback**: funzioni che girano in momenti precisi. `before_tool_callback` gira **prima di ogni strumento**:
- se restituisce `None` → via libera, lo strumento gira;
- se restituisce un **dict** → lo strumento **non gira**, e il modello riceve quel dict come risultato.

✍️ **Tocca a te:** le regole del guardrail. Cosa consideri un dato personale? Quante proposte al massimo?

In [ ]:
# ✍️ TOCCA A TE: le regole del guardrail. Cosa blocchi? Che limite metti?
MAX_PROPOSTE = 3
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
TELEFONO = re.compile(r"\+?\d[\d\s.-]{7,}\d")

def contains_personal_data(args: dict) -> bool:
    testo = " ".join(str(v) for v in args.values())
    return bool(EMAIL.search(testo) or TELEFONO.search(testo))

def guardrail(tool, args, tool_context):
    """Gira PRIMA di ogni strumento. Se restituisce un dict, lo strumento NON viene eseguito."""
    if tool.name == "propose_topic":
        if contains_personal_data(args):
            return {"status": "rifiutato", "motivo": "Niente dati personali (email, telefono) nelle proposte."}
        if tool_context.state.get("proposals_count", 0) >= MAX_PROPOSTE:
            return {"status": "rifiutato", "motivo": f"Massimo {MAX_PROPOSTE} proposte per conversazione."}
    return None  # None = via libera, lo strumento gira normalmente

In [ ]:
pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events, get_faq, get_news, propose_topic],
    before_tool_callback=guardrail,
)
runner = InMemoryRunner(agent=pota_bot)

await chiedi(runner, "Propongo un tema: RAG in azienda. Scrivetemi a mario.rossi@example.com")

🔍 **Osserva:** `[Tool result: {'status': 'rifiutato', ...}]`. Il blocco sta nel **codice**, non nel prompt. **Il prompt si può aggirare, il codice no.**

💼 **In azienda:** è questo che serve per mandare un agente in produzione: privacy (GDPR), limiti di spesa, azioni che richiedono approvazione.

🚀 **Sfida:**
1. Prova ad aggirare la regex: *"mario punto rossi chiocciola example punto com"*. Passa? Cosa servirebbe in produzione?
2. Fai 4 proposte pulite di fila: la quarta viene bloccata?

---
# 🟢 Step 6: Controllo, cosa è successo davvero

🎯 **Obiettivo:** leggere la "scatola nera" dell'agente.

📖 **Capire.** Ogni cosa che succede in una sessione ADK è un **evento**: la tua domanda, ogni decisione del modello, ogni chiamata a uno strumento, ogni risposta. Leggerli è l'**osservabilità**: sapere cosa ha fatto l'agente, perché, e quanto è costato. Questo step **non chiama il modello**: legge solo quello che è già successo.

In [ ]:
def mostra_traccia(eventi):
    """Riassume una sessione: strumenti usati, blocchi e chiamate al modello."""
    chiamate_modello, token = 0, 0
    for e in eventi:
        if e.usage_metadata:
            chiamate_modello += 1
            token += e.usage_metadata.total_token_count or 0
        for fc in e.get_function_calls():
            print(f"🔧 {e.author} chiama {fc.name}({fc.args})")
        for fr in e.get_function_responses():
            print(f"   ↳ risultato: {fr.response}")
    print(f"\n📊 Chiamate al modello: {chiamate_modello} · Token totali: {token}")

sessione = await runner.session_service.get_session(
    app_name=runner.app_name, user_id="debug_user_id", session_id="workshop")
mostra_traccia(sessione.events)

🔍 **Osserva:** ogni passo è tracciabile. Anche il tentativo bloccato dal guardrail.

💼 **In azienda:** *quanto costa?* e *posso verificare cosa ha fatto?* sono le due domande che un CTO fa prima di dire sì a un agente.

🚀 **Sfida:** con i numeri qui sopra, stima quante chiamate farebbero **1.000 utenti al giorno** con 3 domande ciascuno. Poi cerca il prezzo per token del tuo modello: quanto costerebbe al mese?

---
## ✅ Checkpoint: hai un agente funzionante!

Ha i 4 ingredienti:
1. **Istruzioni**: chi è e come parla.
2. **Strumenti**: eventi, FAQ, notizie, proposte.
3. **Stato**: si ricorda le proposte.
4. **Guardrail**: blocca dati personali e troppe proposte, nel codice.

*Un modello risponde. Un agente risolve.*

---
# 🟡 Extra: il digest della settimana

Hai finito il Base? Qui costruiamo quello che la community ha chiesto nel sondaggio: **un digest settimanale di notizie**, con dati freschi da Google Search.

🛟 **Salta qui**: sei rimasto indietro o sei appena arrivato? Esegui la cella qui sotto: ricostruisce tutto fino all'inizio dello step Extra.
*(Serve comunque aver eseguito il Setup. Se hai già fatto gli step precedenti, puoi saltarla.)*

In [ ]:
#@title 🛟 Salta qui: tutto pronto per lo step Extra
from typing import Literal
import re
from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import ToolContext
# 📦 I dati della community: stanno nei file JSON del repo, in pota_bot/data/.
# Aprili dalla barra a sinistra (📁) e modificali come vuoi, poi riesegui questa cella.
import json
from pathlib import Path

DATA_DIR = Path("pota_bot/data")
EVENTS = json.loads((DATA_DIR / "events.json").read_text(encoding="utf-8"))
FAQ = json.loads((DATA_DIR / "faq.json").read_text(encoding="utf-8"))
NEWS = json.loads((DATA_DIR / "news.json").read_text(encoding="utf-8"))  # le 6 categorie del sondaggio
print(f"📦 {len(EVENTS)} eventi, {len(FAQ)} FAQ, notizie in {len(NEWS)} categorie")
# ✍️ TOCCA A TE: questa è la "personalità" del bot. Riscrivila a modo tuo!
INSTRUCTION = """
Sei Pota Bot, l'assistente della community GDG Brescia (Google Developer Group).

Tono: diretto, pratico e amichevole, con un pizzico di bresciano ("Pota!"). Risposte brevi.

Regole:
- Per i fatti (date, luoghi, notizie) usa SEMPRE gli strumenti che hai. Non inventare.
- Se non hai l'informazione, dillo chiaramente.
- Rispondi in italiano.
"""
def get_upcoming_events() -> dict:
    """Restituisce gli eventi di GDG Brescia (passati e futuri) con data, titolo e luogo."""
    return {"events": EVENTS}
def get_faq(topic: str) -> dict:
    """✍️ TOCCA A TE: riscrivi questa docstring. È quello che il modello legge per decidere se usare lo strumento.

    Cerca nelle domande frequenti della community GDG Brescia.

    Args:
        topic: l'argomento della domanda in poche parole, ad es. "costi" o "proporre un talk".
    """
    topic = topic.lower()  # confronto per "radici" di parola: "cost" trova costo, costi, costa
    trovate = [f for f in FAQ if any(p in topic for p in f["parole_chiave"])]
    if not trovate:
        return {"trovate": [], "argomenti_disponibili": [f["domanda"] for f in FAQ]}
    return {"trovate": [{"domanda": f["domanda"], "risposta": f["risposta"]} for f in trovate]}
Category = Literal["eventi_e_opportunita", "modelli_ai", "strumenti_dev",
                   "ai_business", "ecosistema_google", "paper"]

def get_news(category: Category) -> dict:
    """Restituisce le notizie della settimana su AI e Google per una categoria.

    Args:
        category: la categoria più adatta alla domanda. Le categorie sono quelle
            scelte dalla community nel sondaggio Telegram.
    """
    return {"category": category, "news": NEWS[category]}
def propose_topic(title: str, reason: str, tool_context: ToolContext) -> dict:
    """Registra una proposta di tema per un prossimo evento di GDG Brescia.

    Args:
        title: il tema proposto, in poche parole.
        reason: perché interessa alla community.
    """
    # ✍️ TOCCA A TE: cosa vuoi salvare di ogni proposta? (es. anche la data, o una categoria)
    proposals = tool_context.state.get("proposals", [])
    proposals.append({"title": title, "reason": reason})
    tool_context.state["proposals"] = proposals
    tool_context.state["proposals_count"] = len(proposals)
    return {"status": "salvata", "totale_proposte": len(proposals)}
# ✍️ TOCCA A TE: le regole del guardrail. Cosa blocchi? Che limite metti?
MAX_PROPOSTE = 3
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
TELEFONO = re.compile(r"\+?\d[\d\s.-]{7,}\d")

def contains_personal_data(args: dict) -> bool:
    testo = " ".join(str(v) for v in args.values())
    return bool(EMAIL.search(testo) or TELEFONO.search(testo))

def guardrail(tool, args, tool_context):
    """Gira PRIMA di ogni strumento. Se restituisce un dict, lo strumento NON viene eseguito."""
    if tool.name == "propose_topic":
        if contains_personal_data(args):
            return {"status": "rifiutato", "motivo": "Niente dati personali (email, telefono) nelle proposte."}
        if tool_context.state.get("proposals_count", 0) >= MAX_PROPOSTE:
            return {"status": "rifiutato", "motivo": f"Massimo {MAX_PROPOSTE} proposte per conversazione."}
    return None  # None = via libera, lo strumento gira normalmente

pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events, get_faq, get_news, propose_topic],
    before_tool_callback=guardrail,
)
runner = InMemoryRunner(agent=pota_bot)
print("🛟 Sei allineato all'inizio dello step Extra. Puoi continuare da qui.")

## E1: Il digest con Google Search

🎯 **Obiettivo:** un agente che cerca notizie fresche e scrive un digest pronto per Telegram.

📖 **Capire.** Finora le notizie erano scritte da noi dentro `NEWS`. `google_search` è uno strumento **esterno** e già pronto in ADK: l'agente cerca sul web e ottiene risultati di oggi, con le fonti. È anche lo strumento più **caro** (più chiamate) e più **imprevedibile**: per questo sta qui e non nel Base.

In [ ]:
from google.adk.tools.google_search_tool import GoogleSearchTool

# Le 3 categorie più votate nel sondaggio.
CATEGORIE_DIGEST = ["eventi, hackathon e opportunità", "nuovi modelli AI", "strumenti per sviluppatori"]

# ✍️ TOCCA A TE: il formato del digest. Lunghezza, tono, cosa includere o escludere.
DIGEST_INSTRUCTION = f"""
Sei Pota Bot e scrivi il digest settimanale per il gruppo Telegram di GDG Brescia.
Categorie: {", ".join(CATEGORIE_DIGEST)}.
Per ogni categoria cerca con google_search le novità degli ultimi 7 giorni.

Formato:
- Massimo 5 notizie in tutto, una riga ciascuna, ognuna con il link alla fonte.
- Niente hype e niente superlativi: fatti, e perché contano.
- Chiudi con una riga "Pota!" di saluto.
"""

digest_agent = Agent(
    name="pota_digest",
    model=MODEL,
    instruction=DIGEST_INSTRUCTION,
    tools=[GoogleSearchTool(bypass_multi_tools_limit=True)],
)
digest_runner = InMemoryRunner(agent=digest_agent)
await chiedi(digest_runner, "Scrivi il digest di questa settimana.", sessione="digest")

🔍 **Osserva:** le notizie sono fresche? Le fonti sono affidabili? Ha rispettato il formato?

💼 **In azienda:** monitoraggio di mercato, rassegna stampa, analisi dei competitor. Lavoro ripetitivo che un agente fa ogni lunedì mattina.

⚠️ *Se `google_search` non funziona con la tua chiave, usa `get_news` al suo posto: stesso digest, con le notizie scritte da noi.*

## E2: Guardrail "fonti obbligatorie"

🎯 **Obiettivo:** controllare la **risposta** del modello, non solo gli strumenti.

📖 **Capire.** `after_model_callback` gira **dopo** che il modello ha risposto. Se restituisce una nuova risposta, **sostituisce** quella del modello. Qui la usiamo per rifiutare un digest in cui qualche notizia non ha il link alla fonte.

In [ ]:
from google.adk.models.llm_response import LlmResponse
from google.genai import types

def richiedi_fonti(callback_context, llm_response):
    """Rifiuta il digest se una riga-notizia non ha un link."""
    if not llm_response.content or not llm_response.content.parts:
        return None
    testo = "".join(p.text or "" for p in llm_response.content.parts)
    # ✍️ TOCCA A TE: come riconosci una "riga-notizia"? Qui: righe che iniziano con - o •
    notizie = [r for r in testo.splitlines() if r.strip().startswith(("-", "•"))]
    senza_fonte = [r for r in notizie if "http" not in r]
    if senza_fonte:
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(
            text=f"🚫 Digest rifiutato: notizie senza fonte = {len(senza_fonte)}. Riprova chiedendo i link.")]))
    return None

digest_agent = Agent(
    name="pota_digest",
    model=MODEL,
    instruction=DIGEST_INSTRUCTION,
    tools=[GoogleSearchTool(bypass_multi_tools_limit=True)],
    after_model_callback=richiedi_fonti,
)
digest_runner = InMemoryRunner(agent=digest_agent)
await chiedi(digest_runner, "Scrivi il digest di questa settimana.", sessione="digest")

🔍 **Osserva:** prova a togliere dall'istruzione la richiesta dei link. Cosa succede?

💼 **In azienda:** verificabilità. Un'informazione senza fonte, in un report per il management, è un rischio.

## E3: Guardrail "fuori tema"

🎯 **Obiettivo:** fermare le richieste fuori tema **prima** che arrivino al modello.

📖 **Capire.** `before_model_callback` gira **prima** di chiamare il modello. Se restituisce una risposta, il modello **non viene chiamato**: è un controllo che fa anche **risparmiare** chiamate. Qui usiamo una lista di parole; in produzione si usa spesso un piccolo modello che classifica la richiesta.

In [ ]:
# ✍️ TOCCA A TE: di cosa Pota Bot NON deve parlare?
FUORI_TEMA = ["calcio", "politica", "oroscopo", "ricetta"]

def blocca_fuori_tema(callback_context, llm_request):
    ultimo = llm_request.contents[-1] if llm_request.contents else None
    testo = "".join(p.text or "" for p in (ultimo.parts if ultimo else [])).lower()
    if any(parola in testo for parola in FUORI_TEMA):
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(
            text="Pota, di questo non parlo! Chiedimi di eventi, AI o della community.")]))
    return None

pota_bot = Agent(
    name="pota_bot",
    model=MODEL,
    instruction=INSTRUCTION,
    tools=[get_upcoming_events, get_faq, get_news, propose_topic],
    before_tool_callback=guardrail,
    before_model_callback=blocca_fuori_tema,
)
runner = InMemoryRunner(agent=pota_bot)
await chiedi(runner, "Chi vince il campionato di calcio quest'anno?", sessione="fuori_tema")

🔍 **Osserva:** la risposta arriva subito, e con `mostra_traccia` vedresti **zero chiamate al modello**.

💼 **In azienda:** un assistente clienti non deve dare consigli legali o medici. E ogni richiesta bloccata prima del modello è una chiamata risparmiata.

---
# 🔵 Avanzato: da fare a casa

## A1: Collegalo a Telegram

🎯 **Obiettivo:** parlare con il **tuo** Pota Bot da Telegram.

1. Su Telegram apri [@BotFather](https://t.me/BotFather) → `/newbot` → scegli nome e username → copia il **token**.
2. Mettilo nei Secrets di Colab come `TELEGRAM_BOT_TOKEN` (con *Accesso notebook* attivo).
3. Esegui le celle qui sotto e scrivi al tuo bot.

📖 **Capire.** Ogni chat Telegram diventa una **sessione** ADK: il bot si ricorda la conversazione per ogni chat, separatamente.

⚠️ **Il bot vive finché questa scheda di Colab è aperta.** Per tenerlo acceso 24/7 serve un deploy su un server (è nella roadmap del repo).

In [ ]:
%pip install -q python-telegram-bot==22.8

In [ ]:
from telegram import Update
from telegram.ext import Application, MessageHandler, ContextTypes, filters

TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
tg_runner = InMemoryRunner(agent=pota_bot)

async def rispondi(update: Update, context: ContextTypes.DEFAULT_TYPE):
    chat_id = str(update.effective_chat.id)
    servizio = tg_runner.session_service
    if await servizio.get_session(app_name=tg_runner.app_name, user_id=chat_id, session_id=chat_id) is None:
        await servizio.create_session(app_name=tg_runner.app_name, user_id=chat_id, session_id=chat_id)
    messaggio = types.Content(role="user", parts=[types.Part(text=update.message.text)])
    risposta = ""
    async for evento in tg_runner.run_async(user_id=chat_id, session_id=chat_id, new_message=messaggio):
        if evento.is_final_response() and evento.content and evento.content.parts:
            risposta = "".join(p.text or "" for p in evento.content.parts)
    await update.message.reply_text(risposta or "Pota, non ho una risposta.")

# In Colab NON si usa app.run_polling(): il notebook ha già un "event loop" attivo.
app = Application.builder().token(TELEGRAM_BOT_TOKEN).build()
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, rispondi))
await app.initialize()
await app.start()
await app.updater.start_polling()
print("✅ Il bot è acceso: scrivigli su Telegram! (per spegnerlo, esegui la cella qui sotto)")

In [ ]:
#@title ⏹️ Spegni il bot Telegram
await app.updater.stop()
await app.stop()
await app.shutdown()
print("⏹️ Bot spento.")

## A2: Multi-agente

🎯 **Obiettivo:** dividere il lavoro tra agenti specializzati.

📖 **Capire.** Invece di un agente che fa tutto, un **coordinatore** smista le richieste a due **sub-agenti**, ognuno con i suoi strumenti. Il coordinatore sceglie a chi passare la palla leggendo la loro `description`: è come la docstring, ma per gli agenti.

In [ ]:
news_agent = Agent(
    name="news_agent",
    model=MODEL,
    description="Si occupa di notizie e novità su AI e Google.",
    instruction="Rispondi sulle notizie usando get_news. Sii breve e cita le fonti.",
    tools=[get_news],
)
community_agent = Agent(
    name="community_agent",
    model=MODEL,
    description="Si occupa di eventi, domande frequenti e proposte di temi della community GDG Brescia.",
    instruction=INSTRUCTION,
    tools=[get_upcoming_events, get_faq, propose_topic],
    before_tool_callback=guardrail,
)
coordinatore = Agent(
    name="pota_coordinatore",
    model=MODEL,
    instruction="Sei Pota Bot. Passa ogni richiesta all'agente più adatto.",
    sub_agents=[news_agent, community_agent],
)
multi_runner = InMemoryRunner(agent=coordinatore)
await chiedi(multi_runner, "Quando è il prossimo evento? E che novità ci sono sui modelli AI?", sessione="multi")

🔍 **Osserva:** `transfer_to_agent` nell'output. È il **passaggio di consegne** tra agenti.

💼 **In azienda:** separare le responsabilità, come in un team. Ogni agente ha pochi strumenti e un compito chiaro, quindi è più facile da controllare.

## A3: Contribuisci a Pota Bot

Pota Bot è **open source** e il prossimo passo è farlo vivere davvero nel gruppo Telegram della community, con il digest settimanale.

- **Repo:** [github.com/GDGBrescia/pota-bot](https://github.com/GDGBrescia/pota-bot). La cartella `pota_bot/` contiene lo stesso codice di questo notebook, organizzato in file.
- **Come contribuire:** *fork* → modifica → *pull request*. Trovi i dettagli in `CONTRIBUTING.md`.
- **Da dove iniziare:** le issue con l'etichetta `good first issue`, ad esempio:
  - aggiungi una FAQ mancante;
  - migliora il guardrail dei dati personali (codice fiscale, IBAN…);
  - aggiungi uno strumento `get_slides(event)` con i link ai materiali;
  - scrivi un test per il guardrail.

🎉 **La prima pull request della community la festeggiamo al prossimo evento.**

---
### Grazie per essere stati con noi! Pota! 🙌

- Gruppo Telegram: <!-- ⏳ link -->
- Eventi: [Bevy GDG Brescia](https://gdg.community.dev/gdg-brescia/) <!-- ⏳ verificare URL -->
- 🔒 Se hai mostrato la tua chiave API a qualcuno, **revocala** in AI Studio.